# 01 · Shared Data Cleaning

**Team p1zz@:** Sean Qin, Daniel Cai, Surya Venkataraman, Anjeli Noémi (Mimi)

This notebook turns the raw September 2026 Divvy trip file into the cleaned tables every member's analysis starts from.
Run it top to bottom once; it writes its outputs to `data/processed/`.

| Section | Owner (per proposal) |
|---|---|
| 2 Timestamps and day type | Mimi |
| 3 Duplicates | Surya |
| 4 Invalid durations | Surya / Mimi |
| 5 Rider categories | Daniel |
| 6 Station IDs and coverage | Sean |
| 7 Apply exclusions | all (shared validation) |
| 8 Station events and station-day table | Sean |
| 9 Weather | Mimi |
| 10 Validation checks | all |

**Outputs**

- `trips_clean.csv`: one row per kept ride, with parsed times and helper columns
- `exclusions.csv`: how many rides each cleaning rule removed
- `station_events.csv`: one row per departure or arrival at a known station (long format)
- `station_day.csv`: one row per station × day, with morning and later flows and the recovery ratio
- `station_day_by_rider.csv`: same flows split by member/casual (for Daniel)

## 0 · Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path("../data/raw/202609-divvy-tripdata.csv")
OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

# Study window and definitions (from the proposal)
MONTH_START = pd.Timestamp("2026-09-01")
MONTH_END = pd.Timestamp("2026-10-01")   # exclusive
MORNING = (6, 10)    # 6:00-9:59 a.m.
LATER = (10, 20)     # 10:00 a.m.-7:59 p.m.
LABOR_DAY = pd.Timestamp("2026-09-07")

# Cleaning thresholds (team defaults; change here to run sensitivity checks)
MIN_DURATION_SEC = 60          # Divvy's own rule: shorter trips are false starts / re-docks
MAX_DURATION_SEC = 24 * 3600   # longer trips are almost all lost or stolen bikes

## 1 · Load the raw data

In [2]:
raw = pd.read_csv(RAW)
print(raw.shape)
raw.head()

(770562, 13)


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,B3C076B2D6BAC911,electric_bike,2026-09-01 19:28:37.087,2026-09-01 19:33:03.047,NaN,NaN,NaN,NaN,41.87,-87.67,41.87,-87.66,member
1,468834E54297FB3F,electric_bike,2026-09-24 20:38:01.486,2026-09-24 20:44:54.382,NaN,NaN,NaN,NaN,41.91,-87.70,41.90,-87.71,member
2,276E6265CF1F0238,electric_bike,2026-09-24 20:01:27.138,2026-09-24 20:08:32.366,NaN,NaN,NaN,NaN,41.90,-87.71,41.91,-87.70,member
3,F297F75928C557B7,electric_bike,2026-09-29 21:20:03.900,2026-09-29 21:42:29.179,NaN,NaN,NaN,NaN,41.96,-87.68,41.92,-87.72,member
4,0C21D327353C6D73,electric_bike,2026-09-30 13:23:43.088,2026-09-30 13:27:10.920,NaN,NaN,NaN,NaN,41.79,-87.60,41.79,-87.59,member


In [3]:
overview = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_%": (raw.isna().mean() * 100).round(2),
    "unique": raw.nunique(),
})
overview

,dtype,missing,missing_%,unique
ride_id,str,0,0.00,770562
rideable_type,str,0,0.00,2
started_at,str,0,0.00,770363
ended_at,str,0,0.00,770226
start_station_name,str,160341,20.81,1554
start_station_id,str,160341,20.81,1554
end_station_name,str,170033,22.07,1574
end_station_id,str,170033,22.07,1574
start_lat,float64,0,0.00,1591
start_lng,float64,0,0.00,1586


The four station columns are the only ones with meaningful missingness: about 21% of start stations and 22% of end stations.
The 644 missing end coordinates are a subset of the missing end stations.

## 2 · Timestamps and day type (Mimi)

Timestamps are Chicago local time with fractional seconds. September 2026 has no daylight-saving change, so no time-zone adjustment is needed.

In [4]:
trips = raw.copy()
trips["started_at"] = pd.to_datetime(trips["started_at"], format="mixed")
trips["ended_at"] = pd.to_datetime(trips["ended_at"], format="mixed")
trips["duration_sec"] = (trips["ended_at"] - trips["started_at"]).dt.total_seconds()

print("started_at:", trips["started_at"].min(), "to", trips["started_at"].max())
print("ended_at:  ", trips["ended_at"].min(), "to", trips["ended_at"].max())
print("rides starting before Sept 1:", (trips["started_at"] < MONTH_START).sum())

started_at: 2026-08-31 00:53:20.571000 to 2026-09-30 23:59:38.090000
ended_at:   2026-09-01 00:00:18.076000 to 2026-09-30 23:59:48.043000
rides starting before Sept 1: 51


Divvy files a ride under the month it **ends**, so 51 rides start on Aug 31. Their departures fall outside the study window.
Section 8 counts each event by its own timestamp and keeps only events inside September, so we handle these rides there rather than dropping them now.

In [5]:
def add_day_info(df, time_col, prefix):
    t = df[time_col]
    df[f"{prefix}_date"] = t.dt.normalize()
    df[f"{prefix}_hour"] = t.dt.hour
    return df

trips = add_day_info(trips, "started_at", "start")
trips = add_day_info(trips, "ended_at", "end")

calendar = pd.DataFrame({"date": pd.date_range(MONTH_START, MONTH_END - pd.Timedelta(days=1))})
calendar["weekday"] = calendar["date"].dt.day_name()
calendar["is_labor_day"] = calendar["date"] == LABOR_DAY
calendar["day_type"] = np.where(
    calendar["is_labor_day"], "Labor Day",
    np.where(calendar["date"].dt.dayofweek >= 5, "Weekend", "Weekday"),
)
calendar["day_type"].value_counts()

day_type
Weekday      21
Weekend       8
Labor Day     1
Name: count, dtype: int64

## 3 · Duplicates (Surya)

In [6]:
print("duplicate ride_id:", trips["ride_id"].duplicated().sum())
print("fully duplicated rows:", raw.duplicated().sum())

# Near-duplicates: same bike type, rider type, stations, and start/end time to the second
near_key = ["rideable_type", "member_casual", "start_station_id", "end_station_id"]
tmp = trips.assign(s=trips["started_at"].dt.floor("s"), e=trips["ended_at"].dt.floor("s"))
known_both = tmp["start_station_id"].notna() & tmp["end_station_id"].notna()
near_dup_all = tmp[known_both].duplicated(subset=near_key + ["s", "e"], keep=False)
print("rows in a near-duplicate group:", near_dup_all.sum())

trips["flag_near_duplicate"] = False
trips.loc[tmp[known_both].index, "flag_near_duplicate"] = tmp[known_both].duplicated(
    subset=near_key + ["s", "e"], keep="first"
)
tmp[known_both][near_dup_all].sort_values(["s", "start_station_id"])[
    ["ride_id"] + near_key + ["started_at", "ended_at"]
]

duplicate ride_id: 0


fully duplicated rows: 0


rows in a near-duplicate group: 8


,ride_id,rideable_type,member_casual,start_station_id,end_station_id,started_at,ended_at
73564,6E71B764604F8252,electric_bike,casual,CHI00336,CHI02366,2026-09-06 11:04:44.294,2026-09-06 11:34:52.885
522083,91B648886B53F218,electric_bike,casual,CHI00336,CHI02366,2026-09-06 11:04:44.489,2026-09-06 11:34:52.513
217866,A57FA8A23A1EB5A6,classic_bike,casual,CHI00313,CHI00253,2026-09-10 14:42:02.967,2026-09-10 14:51:18.102
220871,2C1F28CECAC632CB,classic_bike,casual,CHI00313,CHI00253,2026-09-10 14:42:02.990,2026-09-10 14:51:18.384
207336,533A11D301780959,electric_bike,member,CHI00242,CHI01772,2026-09-18 18:05:31.983,2026-09-18 18:15:53.437
207388,2B37F5A509EC2E23,electric_bike,member,CHI00242,CHI01772,2026-09-18 18:05:31.107,2026-09-18 18:15:53.392
387868,59297AB5C6D1D2B7,electric_bike,casual,CHI02383,CHI01747,2026-09-19 18:55:08.406,2026-09-19 19:04:06.699
387884,12FF1A92D6555DB9,electric_bike,casual,CHI02383,CHI01747,2026-09-19 18:55:08.111,2026-09-19 19:04:06.128


There are no repeated ride IDs. Four pairs of rides match on every field down to the second; we keep the first ride in each pair and flag the second as a duplicate.

## 4 · Invalid durations

In [7]:
trips["flag_too_short"] = trips["duration_sec"] < MIN_DURATION_SEC
trips["flag_too_long"] = trips["duration_sec"] > MAX_DURATION_SEC

same_station = trips["start_station_id"].notna() & (trips["start_station_id"] == trips["end_station_id"])
print(f"< {MIN_DURATION_SEC}s: {trips['flag_too_short'].sum():,}"
      f"  (same start and end station: {(trips['flag_too_short'] & same_station).sum():,})")
print(f"> 24h: {trips['flag_too_long'].sum():,}"
      f"  (no end station: {(trips['flag_too_long'] & trips['end_station_id'].isna()).sum():,})")
trips["duration_sec"].describe(percentiles=[.01, .25, .5, .75, .99])

< 60s: 18,610  (same start and end station: 6,349)
> 24h: 633  (no end station: 622)


count    770562.000000
mean        909.488887
std        2917.616594
min           0.070000
1%           17.244610
25%         328.981250
50%         564.786500
75%         976.197000
99%        4840.697840
max       90066.163000
Name: duration_sec, dtype: float64

- **Under 60 s:** Divvy's published rule treats these as false starts or riders re-docking a bike. About a third return to the station they left.
- **Over 24 h:** 622 of 633 have no end station, which matches bikes that were lost or stolen rather than ridden.

Both groups are excluded from flows by default. Change `MIN_DURATION_SEC` / `MAX_DURATION_SEC` in Setup to test sensitivity.

## 5 · Rider categories (Daniel)

In [8]:
print(trips["member_casual"].value_counts(dropna=False), "\n")
print(trips["rideable_type"].value_counts(dropna=False), "\n")
pd.crosstab(trips["member_casual"], trips["rideable_type"], margins=True)

member_casual
member    476442
casual    294120
Name: count, dtype: int64 

rideable_type
electric_bike    580638
classic_bike     189924
Name: count, dtype: int64 



rideable_type,classic_bike,electric_bike,All
member_casual,,,
casual,59677,234443,294120
member,130247,346195,476442
All,189924,580638,770562


Both category columns are complete and have only the expected values, so no recoding is needed.

## 6 · Station IDs and coverage (Sean)

In [9]:
# Each ID should map to exactly one name and vice versa
for side in ["start", "end"]:
    ids, names = f"{side}_station_id", f"{side}_station_name"
    known = trips.dropna(subset=[ids])
    print(f"{side}: {known[ids].nunique():,} ids | ids with >1 name: "
          f"{(known.groupby(ids)[names].nunique() > 1).sum()} | names with >1 id: "
          f"{(known.groupby(names)[ids].nunique() > 1).sum()}")

coverage = trips.groupby("rideable_type").agg(
    rides=("ride_id", "size"),
    start_missing_pct=("start_station_id", lambda x: round(x.isna().mean() * 100, 1)),
    end_missing_pct=("end_station_id", lambda x: round(x.isna().mean() * 100, 1)),
)
coverage

start: 1,554 ids | ids with >1 name: 0 | names with >1 id: 0


end: 1,574 ids | ids with >1 name: 0 | names with >1 id: 0


,rides,start_missing_pct,end_missing_pct
rideable_type,,,
classic_bike,189924,0.0,0.3
electric_bike,580638,27.6,29.2


Station IDs and names line up one-to-one, so we can use the ID as the station key.

**Missing stations come only from e-bikes.** E-bikes can be locked away from a dock, so they have no station.
Because of this, flow analyses count only the endpoints that have a known station, and e-bike flows will be undercounted compared with classic bikes.
Daniel's check within bike types should report this.

## 7 · Apply exclusions

In [10]:
rules = [
    ("Near-duplicate (second copy)", "flag_near_duplicate"),
    (f"Duration < {MIN_DURATION_SEC} s", "flag_too_short"),
    ("Duration > 24 h", "flag_too_long"),
]
remaining = pd.Series(True, index=trips.index)
log = [{"step": "Raw rides", "removed": 0, "remaining": len(trips)}]
for label, col in rules:
    drop = remaining & trips[col]
    remaining &= ~trips[col]
    log.append({"step": label, "removed": int(drop.sum()), "remaining": int(remaining.sum())})
exclusions = pd.DataFrame(log)

trips_clean = trips[remaining].copy()
trips_clean["has_start_station"] = trips_clean["start_station_id"].notna()
trips_clean["has_end_station"] = trips_clean["end_station_id"].notna()
trips_clean["has_both_stations"] = trips_clean["has_start_station"] & trips_clean["has_end_station"]
exclusions

,step,removed,remaining
0,Raw rides,0,770562
1,Near-duplicate (second copy),4,770558
2,Duration < 60 s,18610,751948
3,Duration > 24 h,633,751315


In [11]:
print(f"Kept {len(trips_clean):,} of {len(trips):,} rides ({len(trips_clean)/len(trips):.1%})")
print(f"  with start station: {trips_clean['has_start_station'].mean():.1%}")
print(f"  with end station:   {trips_clean['has_end_station'].mean():.1%}")
print(f"  with both (usable for routes): {trips_clean['has_both_stations'].mean():.1%}")

Kept 751,315 of 770,562 rides (97.5%)
  with start station: 79.9%
  with end station:   78.9%
  with both (usable for routes): 66.6%


## 8 · Station events and station-day table (Sean)

Each ride can produce two **events**: a departure at its start station (at `started_at`) and an arrival at its end station (at `ended_at`).
Each event is counted at its own station and time, and only events between Sept 1 and Sept 30 at a known station are kept.

Definitions from the proposal:

- **Morning net departures** = departures − arrivals, 6:00–9:59 a.m.
- **Later net arrivals** = arrivals − departures, 10:00 a.m.–7:59 p.m.
- **Recovery** = later net arrivals ÷ morning net departures, defined only when morning net departures > 0. A value of 1 or more is full recovery; a negative value means the station kept losing bikes.

In [12]:
cols = ["ride_id", "member_casual", "rideable_type"]
dep = trips_clean.loc[trips_clean["has_start_station"], cols + ["start_station_id", "start_station_name", "started_at"]]
dep = dep.set_axis(cols + ["station_id", "station_name", "time"], axis=1).assign(event="departure")
arr = trips_clean.loc[trips_clean["has_end_station"], cols + ["end_station_id", "end_station_name", "ended_at"]]
arr = arr.set_axis(cols + ["station_id", "station_name", "time"], axis=1).assign(event="arrival")

events = pd.concat([dep, arr], ignore_index=True)
events = events[(events["time"] >= MONTH_START) & (events["time"] < MONTH_END)].copy()
events["date"] = events["time"].dt.normalize()
events["hour"] = events["time"].dt.hour
events["window"] = np.select(
    [events["hour"].between(MORNING[0], MORNING[1] - 1), events["hour"].between(LATER[0], LATER[1] - 1)],
    ["morning", "later"],
    default="other",
)
events["event"].value_counts()

event
departure    600151
arrival      592929
Name: count, dtype: int64

In [13]:
def station_day_flows(ev, extra_keys=()):
    keys = ["station_id", "date", *extra_keys]
    counts = (
        ev[ev["window"] != "other"]
        .groupby(keys + ["window", "event"]).size()
        .unstack(["window", "event"], fill_value=0)
    )
    counts.columns = [f"{w}_{e}s" for w, e in counts.columns]
    for c in ["morning_departures", "morning_arrivals", "later_departures", "later_arrivals"]:
        if c not in counts:
            counts[c] = 0
    counts = counts[["morning_departures", "morning_arrivals", "later_departures", "later_arrivals"]]

    # Fill in every station x day (x group) so days with no activity show as zeros
    stations = ev["station_id"].unique()
    levels = [stations, calendar["date"]] + [ev[k].unique() for k in extra_keys]
    full = pd.MultiIndex.from_product(levels, names=keys)
    out = counts.reindex(full, fill_value=0).reset_index()

    out["morning_net_departures"] = out["morning_departures"] - out["morning_arrivals"]
    out["later_net_arrivals"] = out["later_arrivals"] - out["later_departures"]
    out["has_morning_deficit"] = out["morning_net_departures"] > 0
    out["recovery"] = np.where(
        out["has_morning_deficit"], out["later_net_arrivals"] / out["morning_net_departures"].where(out["has_morning_deficit"]), np.nan
    )
    out["full_recovery"] = out["recovery"] >= 1
    return out

names = events.drop_duplicates("station_id").set_index("station_id")["station_name"]
station_day = station_day_flows(events)
station_day.insert(1, "station_name", station_day["station_id"].map(names))
station_day = station_day.merge(calendar, on="date", how="left")
print(station_day.shape)
station_day.sort_values("morning_net_departures", ascending=False).head()

(47670, 15)


,station_id,station_name,date,morning_departures,morning_arrivals,later_departures,later_arrivals,morning_net_departures,later_net_arrivals,has_morning_deficit,recovery,full_recovery,weekday,is_labor_day,day_type
6262,CHI00281,Clark St & Elm St,2026-09-23,46,13,36,59,33,23,True,0.696970,False,Wednesday,False,Weekday
10558,CHI00604,Wabash Ave & 16th St,2026-09-29,29,3,17,31,26,14,True,0.538462,False,Tuesday,False,Weekday
8453,CHI00450,Indiana Ave & Roosevelt Rd,2026-09-24,26,1,31,43,25,12,True,0.480000,False,Thursday,False,Weekday
6250,CHI00281,Clark St & Elm St,2026-09-11,45,20,37,45,25,8,True,0.320000,False,Friday,False,Weekday
6248,CHI00281,Clark St & Elm St,2026-09-09,32,8,54,58,24,4,True,0.166667,False,Wednesday,False,Weekday


In [14]:
deficit = station_day[station_day["has_morning_deficit"]]
print(f"station-days: {len(station_day):,}")
print(f"with a morning deficit: {len(deficit):,} ({len(deficit)/len(station_day):.1%})")
print(f"  deficit >= 5 bikes: {(deficit['morning_net_departures'] >= 5).sum():,}")
print(f"  full recovery rate: {deficit['full_recovery'].mean():.1%}")
deficit["recovery"].describe(percentiles=[.1, .25, .5, .75, .9])

station-days: 47,670
with a morning deficit: 11,408 (23.9%)
  deficit >= 5 bikes: 2,562
  full recovery rate: 37.6%


count    11408.000000
mean         0.444635
std          2.374568
min        -24.000000
10%         -2.000000
25%         -0.333333
50%          0.384615
75%          1.142857
90%          2.500000
max         55.000000
Name: recovery, dtype: float64

In [15]:
# Same flows split by rider group (Daniel). Group counts sum to the totals above.
station_day_by_rider = station_day_flows(events, extra_keys=("member_casual",))
station_day_by_rider = station_day_by_rider.merge(calendar, on="date", how="left")
station_day_by_rider.head()

,station_id,date,member_casual,morning_departures,morning_arrivals,later_departures,later_arrivals,morning_net_departures,later_net_arrivals,has_morning_deficit,recovery,full_recovery,weekday,is_labor_day,day_type
0,CHI00307,2026-09-01,casual,2,2,6,8,0,2,False,NaN,False,Tuesday,False,Weekday
1,CHI00307,2026-09-01,member,3,9,24,18,-6,-6,False,NaN,False,Tuesday,False,Weekday
2,CHI00307,2026-09-02,casual,3,1,10,7,2,-3,True,-1.500000,False,Wednesday,False,Weekday
3,CHI00307,2026-09-02,member,2,9,21,12,-7,-9,False,NaN,False,Wednesday,False,Weekday
4,CHI00307,2026-09-03,casual,8,5,12,8,3,-4,True,-1.333333,False,Thursday,False,Weekday


## 9 · Weather (Mimi)

*To do:* add Meteostat daily data for one Chicago station (e.g. O'Hare, WMO 72530) for September 2026. Save it as `data/raw/weather_202609.csv` and join it to `station_day` on `date`.
Following the proposal, missing precipitation should stay missing (not 0), any model-filled values should be documented, and a rainy day starts out defined as precipitation > 0.

## 10 · Validation checks

In [16]:
# Every kept departure/arrival inside September appears exactly once in the events table
n_dep = ((trips_clean["has_start_station"]) & trips_clean["started_at"].between(MONTH_START, MONTH_END, inclusive="left")).sum()
n_arr = ((trips_clean["has_end_station"]) & trips_clean["ended_at"].between(MONTH_START, MONTH_END, inclusive="left")).sum()
assert (events["event"] == "departure").sum() == n_dep
assert (events["event"] == "arrival").sum() == n_arr

# Station-day totals match the event counts in each window
for w in ["morning", "later"]:
    for e in ["departure", "arrival"]:
        assert station_day[f"{w}_{e}s"].sum() == ((events["window"] == w) & (events["event"] == e)).sum()

# Rider-group flows add up to the totals (Daniel's decomposition)
flow_cols = ["morning_departures", "morning_arrivals", "later_departures", "later_arrivals"]
by_rider_sum = station_day_by_rider.groupby(["station_id", "date"])[flow_cols].sum()
assert by_rider_sum.equals(station_day.set_index(["station_id", "date"])[flow_cols].loc[by_rider_sum.index])

# Recovery only exists where there is a morning deficit
assert station_day.loc[~station_day["has_morning_deficit"], "recovery"].isna().all()
assert station_day.loc[station_day["has_morning_deficit"], "recovery"].notna().all()
print("All checks passed.")

All checks passed.


## 11 · Save outputs

In [17]:
trips_clean.drop(columns=[c for c in trips_clean if c.startswith("flag_")]).to_csv(OUT / "trips_clean.csv", index=False)
exclusions.to_csv(OUT / "exclusions.csv", index=False)
events.to_csv(OUT / "station_events.csv", index=False)
station_day.to_csv(OUT / "station_day.csv", index=False)
station_day_by_rider.to_csv(OUT / "station_day_by_rider.csv", index=False)

for f in sorted(OUT.glob("*.csv")):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:7.1f} MB")

exclusions.csv                   0.0 MB
station_day.csv                  4.6 MB
station_day_by_rider.csv         7.2 MB
station_events.csv             148.8 MB
trips_clean.csv                176.3 MB
